In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from src.morse_dataset import MorseDataset
from src.preprocessing import preprocessing_spectrograms
from src.morse_tcn import MorseTCN
from src.train_model import train_model
from src.collate_fn import collate_fn, test_collate_fn
from src.encoding_decoding import encode_text, decode_predictions, build_char_idx_dict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
df_train_full = pd.read_csv('data/train/labels.csv')
df_train, df_val = train_test_split(df_train_full, test_size=0.2, random_state=42)
df_train

,filename,text
21753,21753.wav,197
251,00251.wav,86
22941,22941.wav,94176
618,00618.wav,9535
17090,17090.wav,959
...,...,...
29802,29802.wav,07-8860
5390,05390.wav,039638
860,00860.wav,28-7
15795,15795.wav,5448374


In [ ]:
# preprocessing_spectrograms('data/train/labels.csv', 'data/train/', 'data/train_specs/')
# preprocessing_spectrograms('sample_submission.csv', 'data/test/', 'data/test_specs/')

  0%|          | 0/30000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

In [3]:
alphabet = '0123456789- '
char_to_idx, idx_to_char = build_char_idx_dict(alphabet=alphabet)

train_files = ['data/train/' + str(f) for f in df_train['filename'].values]
train_labels = df_train['text'].astype(str).values.tolist()

val_files = ['data/train/' + str(f) for f in df_val['filename'].values]
val_labels = df_val['text'].astype(str).values.tolist()

train_dataset = MorseDataset(train_files, train_labels, char_to_idx)
val_dataset = MorseDataset(val_files, val_labels, char_to_idx)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn
)

In [6]:
num_classes = len(char_to_idx)
model = MorseTCN(num_classes=num_classes).to(device)

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

history = train_model(
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    epochs=20,
    device=device,
    save_path="morse_tcn_best.pt",
)

Epoch 1/20 | train_loss: 7.8883 | val_loss: 2.9066 | best_val_loss: 2.9066
Epoch 2/20 | train_loss: 1.7655 | val_loss: 1.2755 | best_val_loss: 1.2755
Epoch 3/20 | train_loss: 0.7625 | val_loss: 2.3035 | best_val_loss: 1.2755
Epoch 4/20 | train_loss: 0.5734 | val_loss: 1.2624 | best_val_loss: 1.2624
Epoch 5/20 | train_loss: 0.4746 | val_loss: 0.7990 | best_val_loss: 0.7990
Epoch 6/20 | train_loss: 0.4135 | val_loss: 1.7635 | best_val_loss: 0.7990
Epoch 7/20 | train_loss: 0.3762 | val_loss: 1.4699 | best_val_loss: 0.7990
Epoch 8/20 | train_loss: 0.3500 | val_loss: 0.6196 | best_val_loss: 0.6196
Epoch 9/20 | train_loss: 0.3306 | val_loss: 1.3600 | best_val_loss: 0.6196
Epoch 10/20 | train_loss: 0.3119 | val_loss: 0.8833 | best_val_loss: 0.6196
Epoch 11/20 | train_loss: 0.2997 | val_loss: 0.8855 | best_val_loss: 0.6196
Epoch 12/20 | train_loss: 0.2828 | val_loss: 0.8410 | best_val_loss: 0.6196
Epoch 13/20 | train_loss: 0.2738 | val_loss: 0.6077 | best_val_loss: 0.6077
Epoch 14/20 | train_l

In [7]:
submission_df = pd.read_csv("sample_submission.csv")
test_files = ["data/test/" + str(f) for f in submission_df["filename"].values]
test_labels = [""] * len(test_files)
test_dataset = MorseDataset(test_files, test_labels, char_to_idx)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=test_collate_fn,
)

In [8]:
num_classes = len(char_to_idx)
model = MorseTCN(num_classes=num_classes).to(device)

checkpoint = torch.load("morse_tcn_best.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

pred_texts = []

with torch.no_grad():
    for inputs, input_lengths in test_loader:
        inputs = inputs.to(device)

        outputs = model(inputs)

        if outputs.size(0) == inputs.size(0):
            outputs = outputs.permute(1, 0, 2)

        pred_ids = outputs.argmax(dim=2).cpu()

        output_time = outputs.size(0)
        input_time = inputs.size(-1)

        scale = output_time / input_time
        output_lengths = torch.floor(input_lengths.float() * scale).long()
        output_lengths = output_lengths.clamp(min=1, max=output_time)

        for i, length in enumerate(output_lengths):
            ids = pred_ids[:length, i].tolist()
            text = decode_predictions(ids, idx_to_char)
            pred_texts.append(text)

submission_df["text"] = pred_texts
submission_df.to_csv("submission.csv", index=False)